# Proyecto: Clasificación de Imágenes con Transfer Learning

## Objetivo
Clasificar imágenes usando un modelo preentrenado (ResNet) adaptado a nuestro dataset.

## Dataset
CIFAR-10: 60,000 imágenes 32x32 en 10 clases.

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Configuración
BATCH_SIZE = 64
EPOCHS = 20
LR = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Usando: {DEVICE}")

# Semilla para reproducibilidad
torch.manual_seed(42)

## 1. Preparar Datos

In [ ]:
# Transforms
# CIFAR-10 es 32x32, ResNet espera mínimo 224x224
# Opción 1: Resize a 224x224 (más lento pero mejor)
# Opción 2: Adaptar ResNet para 32x32 (más rápido)

transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet stats
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Descargar datos
train_data = datasets.CIFAR10('./data', train=True, download=True, transform=transform_train)
test_data = datasets.CIFAR10('./data', train=False, download=True, transform=transform_test)

# Split train/val
train_size = int(0.9 * len(train_data))
val_size = len(train_data) - train_size
train_data, val_data = torch.utils.data.random_split(train_data, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

# Clases
CLASSES = ['avión', 'auto', 'pájaro', 'gato', 'ciervo', 
           'perro', 'rana', 'caballo', 'barco', 'camión']

print(f"Train: {len(train_data)}")
print(f"Val: {len(val_data)}")
print(f"Test: {len(test_data)}")

## 2. Crear Modelo con Transfer Learning

In [ ]:
# Cargar ResNet18 preentrenado
modelo = models.resnet18(weights='IMAGENET1K_V1')

# Congelar capas del backbone
for param in modelo.parameters():
    param.requires_grad = False

# Descongelar últimas capas (fine-tuning parcial)
for param in modelo.layer4.parameters():
    param.requires_grad = True

# Reemplazar clasificador
num_features = modelo.fc.in_features
modelo.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 10)
)

modelo = modelo.to(DEVICE)

# Contar parámetros
total = sum(p.numel() for p in modelo.parameters())
trainable = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Parámetros totales: {total:,}")
print(f"Parámetros entrenables: {trainable:,} ({100*trainable/total:.1f}%)")

## 3. Configurar Entrenamiento

In [ ]:
# Loss y optimizador
loss_fn = nn.CrossEntropyLoss()

# Learning rates diferentes para backbone y clasificador
optimizer = optim.Adam([
    {'params': modelo.layer4.parameters(), 'lr': LR * 0.1},
    {'params': modelo.fc.parameters(), 'lr': LR}
])

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 4. Entrenar

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    
    return total_loss / total, 100 * correct / total

def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    
    return total_loss / total, 100 * correct / total

In [ ]:
# Entrenamiento
historial = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_acc = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(modelo, train_loader, loss_fn, optimizer, DEVICE)
    val_loss, val_acc = evaluate(modelo, val_loader, loss_fn, DEVICE)
    scheduler.step()
    
    historial['train_loss'].append(train_loss)
    historial['val_loss'].append(val_loss)
    historial['train_acc'].append(train_acc)
    historial['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
    
    # Guardar mejor modelo
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(modelo.state_dict(), 'best_model.pth')
        print(f"  ✓ Nuevo mejor modelo guardado!")

## 5. Evaluar

In [ ]:
# Cargar mejor modelo
modelo.load_state_dict(torch.load('best_model.pth'))

# Evaluar en test
test_loss, test_acc = evaluate(modelo, test_loader, loss_fn, DEVICE)
print(f"\nTest Accuracy: {test_acc:.2f}%")

In [ ]:
# Visualizar curvas
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(historial['train_loss'], label='Train')
axes[0].plot(historial['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(historial['train_acc'], label='Train')
axes[1].plot(historial['val_acc'], label='Val')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizar predicciones
modelo.eval()
images, labels = next(iter(test_loader))
images, labels = images[:16].to(DEVICE), labels[:16]

with torch.no_grad():
    outputs = modelo(images)
    preds = outputs.argmax(1).cpu()

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for idx, ax in enumerate(axes.flatten()):
    img = images[idx].cpu().permute(1, 2, 0)
    img = img * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])
    img = img.clamp(0, 1)
    
    ax.imshow(img)
    color = 'green' if preds[idx] == labels[idx] else 'red'
    ax.set_title(f"Pred: {CLASSES[preds[idx]]}\nReal: {CLASSES[labels[idx]]}", color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Conclusiones

- Transfer Learning permite alcanzar buenos resultados con pocos datos
- Fine-tuning parcial (solo últimas capas) es eficiente
- Learning rates diferentes para backbone y clasificador mejoran resultados